In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np

KeyboardInterrupt: 

In [ ]:
#################################
# Path
#################################
dataset_npz_path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_6channel.npz"
test_path = "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_6channel.npz"
model_path = "model.pth"

epochs = 20

device = None

if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"실행 디바이스: {device}")

In [ ]:
#################################
# 1. Dataset Load Class
#################################

class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
data_np = np.load(dataset_npz_path)
X_np, y_np = data_np["X"], data_np["y"]

In [ ]:
#################################
# 2. Causal Convolution Layer
#################################

class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )
        self.left_pad = (kernel_size - 1) * dilation

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))  # 왼쪽만 패딩
        return super().forward(x)


In [ ]:
#################################
# 3. TCN
#################################

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernal_size=3,  dilation=1, dropout=0.1):
        super().__init__()

        self.conv1 = CausalConv1d(n_inputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.conv3 = CausalConv1d(n_outputs, n_outputs, kernel_size=kernal_size, dilation=dilation)
        
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
      
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        def init_one(layer):
            # weight_norm이면 weight_orig가 진짜 파라미터
            w = getattr(layer, "weight_orig", None)
            if w is None:
                w = layer.weight
            nn.init.kaiming_normal_(w)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

        # conv1, conv2가 무엇이든(래퍼든 상속이든) 일단 'Conv1d 파라미터 가진 최종 모듈'에 적용
        init_one(self.conv1)
        init_one(self.conv2)
        init_one(self.conv3)

        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight)
            if self.downsample.bias is not None:
                nn.init.zeros_(self.downsample.bias)
                
    def forward(self, x):
        # x: (B, C, L)
        out = self.conv1(x)          # CausalConv1d 안에서 패딩 + 오른쪽 잘라내기
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.conv3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        res = x if self.downsample is None else self.downsample(x)
        # CausalConv1d가 길이를 유지하니까 따로 slice 안 해도 됨
        return self.relu(out + res)

In [ ]:
#################################
# 4. TCN
#################################

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channel, kernel_size=3, dropout =0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channel)

        dilation = [1,4,9]

        for i in range(num_levels):
            dilation_size = dilation[i]
            in_channels = num_inputs if i == 0 else num_channel[i-1]
            out_channels = num_channel[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size,  dilation = dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

In [ ]:
#################################
# 5. TCN
#################################

class SeqIDS(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.2):
        super(SeqIDS, self).__init__()

        self.tcn = TemporalConvNet(
            num_inputs=6,          # feature 개수
            num_channel=[32, 64],  # 각 레벨 채널 수
            kernel_size=3,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, x, return_attn=False):
        x = self.tcn(x)
        x = self.dropout(x)
        logits = self.classifier(x)

        if return_attn:
            return logits
        
        return logits

In [ ]:
#################################
# 6. Check NAN
#################################

data_load = np.load(dataset_npz_path)
X_np, y_np= data_load["X"], data_load["y"]

### NAN 값 확인 ###
for i in range(X_np.shape[1]):
        feat_nan = np.isnan(X_np[:, i, :]).sum()
        print(f"Feature {i}의 NaN 개수: {feat_nan}")

        nan_count_X = np.isnan(X_np).sum()
        nan_count_y = np.isnan(y_np).sum()
        print(f"데이터 검사 결과:")
        print(f"   - X 내 NaN 개수: {nan_count_X}개")
        print(f"   - y 내 NaN 개수: {nan_count_y}개")
        if nan_count_X > 0:
            print("주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.")


inf_count = np.isinf(X_np).sum() # 무한대 체크 추가
print(f"🔍 X 내 inf 개수: {inf_count}개")
if inf_count > 0:
    print("경고: 데이터에 inf(무한대)가 포함되어 있습니다. 인코딩 스크립트를 수정하세요.")

In [ ]:
#################################
# 6. First TCN
#################################

full_dataset = LoadDatset(X_np, y_np)

#### Validation Split (80:20) ###
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

### 시드 고정 ###
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")

### Model and DataLoader ###
model1 = SeqIDS(num_classes=5, dropout_rate=0.5).to(device)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### Config Model ###
W_conv1 = model1.tcn.network[0].conv1.weight
other_params = [p for n, p in model1.named_parameters() if "tcn.network.0.conv1.weight" not in n]

optimizer1 = torch.optim.Adam([
    {"params": [W_conv1], "weight_decay": 0},      # conv1은 수동 페널티를 위해 WD 0으로 설정
    {"params": other_params, "weight_decay": 1e-4} # 나머지는 일반적인 WD 적용
    ], lr=1e-4)

scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode='min', factor=0.5, patience=3
)



### data Weight AND LOSS Func ###
weights = torch.tensor([1.0, 1.0, 1.0, 1.0, 2.0]).to(device)
criterion1 = nn.CrossEntropyLoss(weight=weights)

### weight decay ###

decay_map = {
        0: 0.01,  #ID IAT
        1: 0.01,  #Dos 0
        2: 0.001, #Entropy
        3: 0.01,  #Jitter                                                             
        4: 0.01,  #ID hamming
        5: 0.001  #Frequency
    }


def channel_l2_penalty(conv1_weight, decay_map):
    # conv1_weight: (out_ch, in_ch, k)
    pen = 0.0
    for ch, wd in decay_map.items():   # decay_map: {0:...,1:...,...}
        w_ch = conv1_weight[:, ch:ch+1, :]
        pen = pen + wd * (w_ch.pow(2).sum())
    return pen


# [일반화 5] Early Stopping 변수
best_val_loss = float('inf')
best_model_state = None

prev_train_loss = None
prev_val_loss   = None
overfit_wait    = 0
overfit_patience = 5
min_delta = 1e-4        # val loss가 이만큼은 줄어야 "개선"으로 인정
es_wait   = 0   

print("\n Start Training...")

### Train Loop ###
for epoch in range(1, epochs+ 1):
    model1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
        y1 = torch.zeros_like(y_batch)
        y1 = torch.where(y_batch == 1, torch.ones_like(y1), y1)
        y1 = torch.where(y_batch == 2, torch.full_like(y1, 2), y1)

        optimizer1.zero_grad()
        
        logits1 = model1(X_batch)
        loss = criterion1(logits1.permute(0, 2, 1).reshape(-1, 5), y1.reshape(-1))

        W = model1.tcn.network[0].conv1.weight
        loss = loss + channel_l2_penalty(W, decay_map)
        
        loss.backward()
        optimizer1.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model1.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)

            ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
            
            y1v = torch.zeros_like(y_val)
            y1v = torch.where(y_val == 1, torch.ones_like(y1v), y1v)
            y1v = torch.where(y_val == 2, torch.full_like(y1v, 2), y1v)
            
            logits1 = model1(X_val)
            loss = criterion1(logits1.permute(0, 2, 1).reshape(-1, 5), y1v.reshape(-1))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler1.step(avg_val_loss)
    current_lr = optimizer1.param_groups[0]['lr']

    print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")

NameError: name 'np' is not defined

In [ ]:
#################################
# 7. Second TCN
#################################

full_dataset = LoadDatset(X_np, y_np)

#### Validation Split (80:20) ###
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

### 시드 고정 ###
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

print(f"데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")

### Model and DataLoader ###
model2 = SeqIDS(num_classes=5, dropout_rate=0.5).to(device)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

### Config Model ###
W_conv2 = model1.tcn.network[0].conv1.weight
other_params = [p for n, p in model1.named_parameters() if "tcn.network.0.conv1.weight" not in n]

optimizer2 = torch.optim.Adam([
    {"params": [W_conv1], "weight_decay": 0},      # conv1은 수동 페널티를 위해 WD 0으로 설정
    {"params": other_params, "weight_decay": 1e-4} # 나머지는 일반적인 WD 적용
    ], lr=1e-4)

scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode='min', factor=0.5, patience=3
)



### data Weight AND LOSS Func ###
weights = torch.tensor([1.0, 1.0, 1.0, 1.0, 2.0]).to(device)
criterion2 = nn.CrossEntropyLoss(weight=weights)

### weight decay ###

decay_map = {
        0: 0.01,  #ID IAT
        1: 0.01,  #Dos 0
        2: 0.001, #Entropy
        3: 0.01,  #Jitter                                                             
        4: 0.01,  #ID hamming
        5: 0.001  #Frequency
    }


def channel_l2_penalty(conv1_weight, decay_map):
    # conv1_weight: (out_ch, in_ch, k)
    pen = 0.0
    for ch, wd in decay_map.items():   # decay_map: {0:...,1:...,...}
        w_ch = conv1_weight[:, ch:ch+1, :]
        pen = pen + wd * (w_ch.pow(2).sum())
    return pen


# [일반화 5] Early Stopping 변수
best_val_loss = float('inf')
best_model_state = None

prev_train_loss = None
prev_val_loss   = None
overfit_wait    = 0
overfit_patience = 5
min_delta = 1e-4        # val loss가 이만큼은 줄어야 "개선"으로 인정
es_wait   = 0   

print("\n Start Training...")

### Train Loop ###
for epoch in range(1, epochs+ 1):
    model2.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
        y2 = y_batch.clone()
        

        optimizer2.zero_grad()
        
        logits2 = model2(X_batch)
        loss = criterion1(logits2.permute(0, 2, 1).reshape(-1, 5), y1.reshape(-1))

        W = model1.tcn.network[0].conv1.weight
        loss = loss + channel_l2_penalty(W, decay_map)
        
        loss.backward()
        optimizer1.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)

    # --- Validation ---
    model1.eval()
    val_loss = 0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)

            ### y_stage1: 0=OTHER(0/3/4), 1=DoS, 2=Fuzz ###
            
            y1v = torch.zeros_like(y_val)
            y1v = torch.where(y_val == 1, torch.ones_like(y1v), y1v)
            y1v = torch.where(y_val == 2, torch.full_like(y1v, 2), y1v)
            
            logits1 = model1(X_val)
            loss = criterion1(logits1.permute(0, 2, 1).reshape(-1, 5), y1v.reshape(-1))
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    
    scheduler1.step(avg_val_loss)
    current_lr = optimizer1.param_groups[0]['lr']

    print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")